In [3]:
from agent import faq_agent, SearchDeps
from ingest import build_index, load_faq_data
import logfire

logfire.configure()
logfire.instrument_pydantic_ai()

documents = load_faq_data()
index = build_index(documents)
deps = SearchDeps(index=index)

result = await faq_agent.run(
    "How do I run Ollama locally?",
    deps=deps
)
print(result.output)

Logfire project URL: https://logfire-us.pydantic.dev/dim2e3/starter-project

19:31:03.617 faq_agent run
19:31:03.638   chat gpt-5.4-mini
19:31:07.208   running tool: search
19:31:07.308   chat gpt-5.4-mini
You can run Ollama locally by:

1. Installing Ollama from https://ollama.com/download
   - macOS: download the `.pkg`
   - Windows: download the `.msi`
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Starting a model locally:
   ```bash
   ollama run llama3
   ```
   This downloads the model, starts it locally, and opens a chat-like interface.

3. Testing the local server:
   ```bash
   curl http://localhost:11434
   ```
   You should get a response like:
   ```json
   {"models": [...]} 
   ```

4. If you want to use it from Python:
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": your_prompt}]
   )

   print(response['message']['content'])
   ```

If you’d like, I can also help with ru

Question 1: For the query "How do I run Ollama locally?", how many spans does a single agent run produce?

19:31:03.617 faq_agent run
19:31:03.638   chat gpt-5.4-mini
19:31:07.208   running tool: search
19:31:07.308   chat gpt-5.4-mini

4 spans

Answer: 5 (nearest) 

In [7]:
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
import os

print(os.getenv("LOGFIRE_READ_TOKEN"))

pylf_v1_us_2jz8g715bNndR9dNV2spNbmmJvqjMqqTC0cw7VfJwvyz


In [10]:
import os
import requests
import dlt


def unflatten(d):
    if not isinstance(d, dict):
        return d

    result = {}

    for key, value in d.items():
        parts = key.split(".")
        current = result

        for part in parts[:-1]:
            if part not in current:
                current[part] = {}
            current = current[part]

        current[parts[-1]] = value

    return result


@dlt.source
def logfire_source(read_token=None):

    if read_token is None:
        read_token = os.environ["LOGFIRE_READ_TOKEN"]

    url = "https://logfire-api.pydantic.dev/v1/query"

    headers = {
        "Authorization": f"Bearer {read_token}"
    }

    params = {
        "project_id": "dim2e3/starter-project",
        "sql": "SELECT * FROM records"
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    response.raise_for_status()

    data = response.json()

    columns = data.get("columns", [])

    if not columns:
        return

    row_count = len(columns[0]["values"])

    rows = []

    for i in range(row_count):

        row = {}

        for column in columns:

            value = column["values"][i]

            if isinstance(value, dict):
                value = unflatten(value)

            elif isinstance(value, list):
                value = [
                    unflatten(x) if isinstance(x, dict) else x
                    for x in value
                ]

            row[column["name"]] = value

        rows.append(row)


    @dlt.resource(name="logfire_records")
    def logfire_records():
        yield from rows


    return logfire_records

In [12]:
load_info = pipeline.run(
    logfire_source()
)

print(load_info)

2026-07-20 19:50:30,852|[WARNING]|125589|135692591114048|dlt|validate.py|verify_normalized_table:113|In schema `logfire_source`: The following columns in table 'logfire_records' did not receive any data during this load and therefore could not have their types inferred:
  - attributes__model_request_parameters__output_object
  - attributes__model_request_parameters__prompted_output_template
  - attributes__model_request_parameters__thinking
  - deployment_environment
  - exception_message
  - exception_stacktrace
  - exception_type
  - http_method
  - http_response_status_code
  - http_route
  - log_body
  - otel_status_message
  - url_full
  - url_path
  - url_query

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'attributes__model_request_parameters__output_object': {'data_type': 'text'}})

2026-07-20 19:50:

Pipeline logfire_pipeline load step completed in 0.88 seconds
1 load package(s) were loaded to destination duckdb and into dataset agent_traces
The duckdb destination used duckdb:////workspaces/llm-zoomcamp-2026-code/homework/06-dlt-workshop/logfire_pipeline.duckdb location to store data
Load package 1784577030.5082107 is LOADED and contains no failed jobs


In [21]:
import duckdb

conn = duckdb.connect('logfire_pipeline.duckdb')
tables = conn.execute("SELECT COUNT(*) FROM information_schema.tables WHERE table_schema = 'agent_traces';").fetchone()[0]
print(tables)

24


In [22]:
conn.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'agent_traces'").fetchall()

[('logfire_records',),
 ('logfire_records__attributes__gen_ai__input__messages',),
 ('logfire_records__attributes__gen_ai__input__messages__parts',),
 ('logfire_records__attributes__gen_ai__input__messages__parts__result',),
 ('logfire_records__attributes__gen_ai__output__messages',),
 ('logfire_records__attributes__gen_ai__output__messages__parts',),
 ('logfire_records__attributes__gen_ai__response__finish_reasons',),
 ('logfire_records__attributes__gen_ai__system_instructions',),
 ('logfire_records__attributes__gen_ai__tool__call__result',),
 ('logfire_records__attributes__gen_ai__tool__definitions',),
 ('logfire_records__attributes__gen_ai__tool__definitions__parameters__required',),
 ('logfire_records__attributes__logfire__metrics__gen_ai_client_token_usage__details',),
 ('logfire_records__attributes__logfire__metrics__operation_cost__details',),
 ('logfire_records__attributes__logfire__scrubbed',),
 ('logfire_records__attributes__logfire__scrubbed__path',),
 ('logfire_records__att

In [23]:
conn.execute("SELECT column_name FROM information_schema.columns WHERE table_schema = 'agent_traces' AND table_name = 'query' LIMIT 50").fetchall()

[]

In [24]:
conn.execute("""
SELECT *
FROM agent_traces.logfire_records
LIMIT 5
""").df()

,created_at,start_timestamp,end_timestamp,duration,trace_id,span_id,kind,level,span_name,message,...,attributes__gen_ai__request__model,attributes__gen_ai__provider__name,attributes_json_schema__properties__model_request_parameters__type,attributes_json_schema__properties__gen_ai_output_messages__type,attributes_json_schema__properties__gen_ai_input_messages__type,attributes__gen_ai__tool__name,attributes__gen_ai__tool__call__id,attributes__gen_ai__tool__call__arguments__query,attributes_json_schema__properties__gen_ai_tool_call_result__type,attributes_json_schema__properties__gen_ai_tool_call_arguments__type
0,2026-07-20 19:31:12.192613+00:00,2026-07-20 19:31:03.617363+00:00,2026-07-20 19:31:10.971648+00:00,7.354285,019f8102ab41840ef84bb4c2128d06b8,4b2b90222698c3b1,span,9,invoke_agent faq_agent,faq_agent run,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-07-20 19:31:12.192613+00:00,2026-07-20 19:31:07.308418+00:00,2026-07-20 19:31:10.970067+00:00,3.661649,019f8102ab41840ef84bb4c2128d06b8,5bdafc9cbbfec3c8,span,9,chat gpt-5.4-mini,chat gpt-5.4-mini,...,gpt-5.4-mini,openai,object,array,array,NaN,NaN,NaN,NaN,NaN
2,2026-07-20 19:31:08.418021+00:00,2026-07-20 19:31:07.208572+00:00,2026-07-20 19:31:07.304469+00:00,0.095897,019f8102ab41840ef84bb4c2128d06b8,8afceea157c7dddf,span,9,execute_tool search,running tool: search,...,NaN,NaN,NaN,NaN,NaN,search,call_qEmWJlm7PaUc0q7TqI9tIScD,run Ollama locally install use locally Ollama,object,object
3,2026-07-20 19:31:08.418021+00:00,2026-07-20 19:31:03.638361+00:00,2026-07-20 19:31:07.206002+00:00,3.567641,019f8102ab41840ef84bb4c2128d06b8,fbf57c06b6fc6c01,span,9,chat gpt-5.4-mini,chat gpt-5.4-mini,...,gpt-5.4-mini,openai,object,array,array,NaN,NaN,NaN,NaN,NaN


In [25]:
import pandas as pd

# Check columns first
df = conn.execute("""
SELECT *
FROM agent_traces.logfire_records
LIMIT 1
""").df()

print(df.columns.tolist())

['created_at', 'start_timestamp', 'end_timestamp', 'duration', 'trace_id', 'span_id', 'kind', 'level', 'span_name', 'message', 'otel_status_code', 'is_exception', 'otel_scope_name', 'otel_scope_version', 'service_namespace', 'service_name', 'service_version', 'service_instance_id', 'process_pid', 'telemetry_sdk_name', 'telemetry_sdk_language', 'telemetry_sdk_version', 'project_id', 'day', 'otel_resource_attributes__telemetry__sdk__language', 'otel_resource_attributes__telemetry__sdk__name', 'otel_resource_attributes__telemetry__sdk__version', 'otel_resource_attributes__service__name', 'otel_resource_attributes__service__version', 'otel_resource_attributes__service__instance__id', 'otel_resource_attributes__process__pid', 'otel_resource_attributes__process__runtime__description', 'otel_resource_attributes__process__runtime__name', 'otel_resource_attributes__process__runtime__version', 'otel_resource_attributes__os__type', 'otel_resource_attributes__os__version', 'otel_resource_attribute

In [32]:
conn.execute("""
SELECT DISTINCT span_name
FROM agent_traces.logfire_records
ORDER BY span_name
""").fetchall()

[('chat gpt-5.4-mini',), ('execute_tool search',), ('invoke_agent faq_agent',)]

In [33]:
conn.execute("""
SELECT
    trace_id,
    span_name,
    start_timestamp
FROM agent_traces.logfire_records
LIMIT 20
""").df()

,trace_id,span_name,start_timestamp
0,019f8102ab41840ef84bb4c2128d06b8,invoke_agent faq_agent,2026-07-20 19:31:03.617363+00:00
1,019f8102ab41840ef84bb4c2128d06b8,chat gpt-5.4-mini,2026-07-20 19:31:07.308418+00:00
2,019f8102ab41840ef84bb4c2128d06b8,execute_tool search,2026-07-20 19:31:07.208572+00:00
3,019f8102ab41840ef84bb4c2128d06b8,chat gpt-5.4-mini,2026-07-20 19:31:03.638361+00:00


In [34]:
result = conn.execute("""
SELECT trace_id
FROM agent_traces.logfire_records
WHERE span_name LIKE '%faq_agent%'
ORDER BY start_timestamp DESC
LIMIT 1
""").fetchone()

if result is None:
    raise Exception("No agent run found")

trace_id = result[0]

print(trace_id)

019f8102ab41840ef84bb4c2128d06b8


In [35]:
conn.execute("""
SELECT
    span_name,
    trace_id
FROM agent_traces.logfire_records
WHERE span_name LIKE '%gpt%'
""").df()

,span_name,trace_id
0,chat gpt-5.4-mini,019f8102ab41840ef84bb4c2128d06b8
1,chat gpt-5.4-mini,019f8102ab41840ef84bb4c2128d06b8


In [36]:
conn.execute("""
SELECT *
FROM agent_traces.logfire_records__attributes__logfire__metrics__gen_ai_client_token_usage__details
LIMIT 10
""").df()

,total,attributes__gen_ai_operation_name,attributes__gen_ai_provider_name,attributes__gen_ai_request_model,attributes__gen_ai_response_model,attributes__gen_ai_system,attributes__gen_ai_token_type,_dlt_parent_id,_dlt_list_idx,_dlt_id
0,1825,chat,openai,gpt-5.4-mini,gpt-5.4-mini-2026-03-17,openai,input,WnTaCjXRnqXY4Q,0,omDh3V9P96si4g
1,290,chat,openai,gpt-5.4-mini,gpt-5.4-mini-2026-03-17,openai,output,WnTaCjXRnqXY4Q,1,zNliL4b91d9aXw


In [40]:
import duckdb

conn = duckdb.connect("logfire_pipeline.duckdb")

df = conn.execute("""
    SELECT
        trace_id,
        span_name,
        start_timestamp
    FROM agent_traces.logfire_records
    ORDER BY start_timestamp DESC
    LIMIT 50
""").df()

df

,trace_id,span_name,start_timestamp
0,019f8102ab41840ef84bb4c2128d06b8,chat gpt-5.4-mini,2026-07-20 19:31:07.308418+00:00
1,019f8102ab41840ef84bb4c2128d06b8,execute_tool search,2026-07-20 19:31:07.208572+00:00
2,019f8102ab41840ef84bb4c2128d06b8,chat gpt-5.4-mini,2026-07-20 19:31:03.638361+00:00
3,019f8102ab41840ef84bb4c2128d06b8,invoke_agent faq_agent,2026-07-20 19:31:03.617363+00:00


In [48]:
import duckdb

conn = duckdb.connect("logfire_pipeline.duckdb")

# Find the agent trace
trace_id = conn.execute("""
    SELECT trace_id
    FROM agent_traces.logfire_records
    WHERE span_name = 'invoke_agent faq_agent'
    ORDER BY start_timestamp DESC
    LIMIT 1
""").fetchone()[0]

print("Trace ID:", trace_id)


# Get all LLM calls and their input tokens
df = conn.execute("""
    SELECT
        span_name,
        attributes__gen_ai__usage__input_tokens AS input_tokens
    FROM agent_traces.logfire_records
    WHERE trace_id = ?
      AND span_name = 'chat gpt-5.4-mini'
    ORDER BY start_timestamp
""", [trace_id]).df()


print(df)

# Sum input tokens across all LLM calls
total_input_tokens = df["input_tokens"].sum()

print("----------------------------")
print("Total input tokens:", total_input_tokens)
print("----------------------------")

Trace ID: 019f8102ab41840ef84bb4c2128d06b8
           span_name  input_tokens
0  chat gpt-5.4-mini           204
1  chat gpt-5.4-mini          1621
----------------------------
Total input tokens: 1825
----------------------------


Question 3: What is the range of total input token usage for the agent run from Q1? 

Answer: 1500 - 5000

In [42]:
import duckdb

conn = duckdb.connect("logfire_pipeline.duckdb")

token_table = """
agent_traces.logfire_records__attributes__logfire__metrics__gen_ai_client_token_usage__details
"""

print(
    conn.execute(f"""
        DESCRIBE {token_table}
    """).df()
)

print(
    conn.execute(f"""
        SELECT *
        FROM {token_table}
        LIMIT 5
    """).df()
)

                         column_name column_type null   key default extra
0                              total      BIGINT  YES  None    None  None
1  attributes__gen_ai_operation_name     VARCHAR  YES  None    None  None
2   attributes__gen_ai_provider_name     VARCHAR  YES  None    None  None
3   attributes__gen_ai_request_model     VARCHAR  YES  None    None  None
4  attributes__gen_ai_response_model     VARCHAR  YES  None    None  None
5          attributes__gen_ai_system     VARCHAR  YES  None    None  None
6      attributes__gen_ai_token_type     VARCHAR  YES  None    None  None
7                     _dlt_parent_id     VARCHAR   NO  None    None  None
8                      _dlt_list_idx      BIGINT   NO  None    None  None
9                            _dlt_id     VARCHAR   NO  None    None  None
   total attributes__gen_ai_operation_name attributes__gen_ai_provider_name  \
0   1825                              chat                           openai   
1    290                    

In [46]:
conn.execute("""
SELECT *
FROM agent_traces.logfire_records
WHERE span_name = 'chat gpt-5.4-mini'
LIMIT 1
""").df()

,created_at,start_timestamp,end_timestamp,duration,trace_id,span_id,kind,level,span_name,message,...,attributes__gen_ai__request__model,attributes__gen_ai__provider__name,attributes_json_schema__properties__model_request_parameters__type,attributes_json_schema__properties__gen_ai_output_messages__type,attributes_json_schema__properties__gen_ai_input_messages__type,attributes__gen_ai__tool__name,attributes__gen_ai__tool__call__id,attributes__gen_ai__tool__call__arguments__query,attributes_json_schema__properties__gen_ai_tool_call_result__type,attributes_json_schema__properties__gen_ai_tool_call_arguments__type
0,2026-07-20 19:31:12.192613+00:00,2026-07-20 19:31:07.308418+00:00,2026-07-20 19:31:10.970067+00:00,3.661649,019f8102ab41840ef84bb4c2128d06b8,5bdafc9cbbfec3c8,span,9,chat gpt-5.4-mini,chat gpt-5.4-mini,...,gpt-5.4-mini,openai,object,array,array,None,None,None,None,None


In [47]:
import duckdb

conn = duckdb.connect("logfire_pipeline.duckdb")

# Show columns related to token usage
columns = conn.execute("""
    DESCRIBE agent_traces.logfire_records
""").df()

columns[
    columns["column_name"].str.contains(
        "token|usage",
        case=False,
        na=False
    )
]

,column_name,column_type,null,key,default,extra
43,attributes__logfire__metrics__gen_ai_client_to...,BIGINT,YES,None,None,None
46,attributes__gen_ai__aggregated_usage__input_to...,BIGINT,YES,None,None,None
47,attributes__gen_ai__aggregated_usage__output_t...,BIGINT,YES,None,None,None
63,attributes__gen_ai__usage__input_tokens,BIGINT,YES,None,None,None
64,attributes__gen_ai__usage__output_tokens,BIGINT,YES,None,None,None
